# Source



[ESG경제](https://www.esgeconomy.com/news/articleView.html?idxno=6361&utm_source=chatgpt.com)

> 주거용 건물의 경우, EU 회원국은 2030년까지 평균 기본 에너지 사용량을 최소 16% 이상, 2035년까지는 20~22% 이상 감소시키는 조치를 마련해야 한다. 회원국은 2030년까지 성능이 가장 낮은 비주거용 건물 중 16%를 개조해야 하며, 2033년까지는 26%를 최소 에너지 성능 기준에 맞게 개조해야 한다.

In [1]:
import xml.etree.ElementTree as ET

# ---------------------------
# 설정: 입력/출력 파일명
# ---------------------------
in_path = "../../input/policy/korea-2035/buildings/green_remodeling_cp.xml"
out_path = "../../input/policy/korea-2035/buildings/green_remodeling_ep.xml"

# 감소율(2025 대비)
decrease_2030 = 0.16
decrease_2035 = 0.22

# 대상 연도
base_year = "2025"
year_2030 = "2030"
year_2035 = "2035"

# ---------------------------
# XML 읽기
# ---------------------------
tree = ET.parse(in_path)
root = tree.getroot()

# ---------------------------
# 2025 shell-conductance 기반으로 2030/2035 추가
# ---------------------------
count_updated = 0
count_skipped_no_2025 = 0
count_errors = 0

# 모든 shell-conductance를 훑되, year="2025"인 것만 기준으로 삼음
for sc_2025 in root.findall(".//shell-conductance[@year='2025']"):
    parent = sc_2025.getparent() if hasattr(sc_2025, "getparent") else None
    # xml.etree.ElementTree는 기본적으로 getparent()가 없음.
    # 그래서 부모를 직접 찾아야 함.
    # 아래는 안전하게 부모를 찾아내는 방식.

# 부모 찾기용: (child -> parent) 매핑 만들기
parent_map = {c: p for p in root.iter() for c in p}

count_updated = 0
count_skipped_no_2025 = 0
count_errors = 0

for sc_2025 in root.findall(".//shell-conductance[@year='2025']"):
    try:
        base_val = float((sc_2025.text or "").strip())
    except Exception:
        count_errors += 1
        continue

    parent = parent_map.get(sc_2025, None)
    if parent is None:
        count_errors += 1
        continue

    val_2030 = base_val * (1.0 - decrease_2030)
    val_2035 = base_val * (1.0 - decrease_2035)

    # 동일 parent 아래에 year=2030/2035가 이미 있으면 갱신, 없으면 추가
    sc_2030 = parent.find("./shell-conductance[@year='2030']")
    if sc_2030 is None:
        sc_2030 = ET.SubElement(parent, "shell-conductance", {"year": year_2030})
    sc_2030.text = f"{val_2030:.7f}"

    sc_2035 = parent.find("./shell-conductance[@year='2035']")
    if sc_2035 is None:
        sc_2035 = ET.SubElement(parent, "shell-conductance", {"year": year_2035})
    sc_2035.text = f"{val_2035:.7f}"

    count_updated += 1

# ---------------------------
# 보기 좋게 들여쓰기 (Python 3.9+)
# ---------------------------
try:
    ET.indent(tree, space="  ", level=0)
except Exception:
    # ET.indent가 없는 구버전이면 그냥 저장(기능상 문제 없음)
    pass

# ---------------------------
# 새 파일로 저장
# ---------------------------
tree.write(out_path, encoding="utf-8", xml_declaration=True)

print(f"Done. Updated nodes: {count_updated}, parse errors: {count_errors}")
print(f"Saved to: {out_path}")

Done. Updated nodes: 11, parse errors: 0
Saved to: ../../input/policy/korea-2035/buildings/green_remodeling_ep.xml
